In [ ]:
# Visualize sampled E-field CSV output from FieldSolver
# Works with files named like: E_field_samples_512x512_step_000010.csv

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional: set to a specific file name, otherwise the latest step file is used.
TARGET_FILE = None  # e.g. "E_field_samples_512x512_step_000010.csv"
PATTERN = "E_field_samples_512x512_step_*.csv"


def pick_file(target_file=None, pattern=PATTERN):
    if target_file is not None:
        p = Path(target_file)
        if not p.exists():
            raise FileNotFoundError(f"File not found: {p}")
        return p

    files = sorted(Path('.').glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files found matching: {pattern}")

    # Prefer highest numeric step suffix
    def step_num(path):
        m = re.search(r"step_(\d+)\.csv$", path.name)
        return int(m.group(1)) if m else -1

    return max(files, key=step_num)


csv_path = pick_file(TARGET_FILE)
print(f"Reading: {csv_path}")

df = pd.read_csv(csv_path)
required = {"i", "j", "x", "y", "Ex", "Ey"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

# Infer grid shape
nx = int(df["i"].max()) + 1
ny = int(df["j"].max()) + 1
if len(df) != nx * ny:
    raise ValueError(f"Expected {nx*ny} rows from i/j grid, got {len(df)}")

# Sort for consistent reshape: j major then i
df = df.sort_values(["j", "i"], kind="mergesort")

x = df["x"].to_numpy().reshape(ny, nx)
y = df["y"].to_numpy().reshape(ny, nx)
Ex = df["Ex"].to_numpy().reshape(ny, nx)
Ey = df["Ey"].to_numpy().reshape(ny, nx)
Emag = np.sqrt(Ex**2 + Ey**2)

step_val = int(df["step"].iloc[0]) if "step" in df.columns else None
step_label = f"step {step_val}" if step_val is not None else csv_path.name

# Plot |E| heatmap + quiver (downsampled)
fig, ax = plt.subplots(figsize=(8, 7))

extent = [x.min(), x.max(), y.min(), y.max()]
im = ax.imshow(
    Emag,
    origin="lower",
    extent=extent,
    aspect="equal",
    cmap="viridis"
)

# Downsample arrows for readability
qskip = max(1, nx // 32)
ax.quiver(
    x[::qskip, ::qskip],
    y[::qskip, ::qskip],
    Ex[::qskip, ::qskip],
    Ey[::qskip, ::qskip],
    color="white",
    alpha=0.75,
    pivot="mid",
    scale=None
)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("|E|")
ax.set_title(f"Electric field magnitude and direction ({step_label})")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.tight_layout()
plt.show()

print(f"Grid: nx={nx}, ny={ny}")
print(f"|E| min/max: {Emag.min():.6e}, {Emag.max():.6e}")